# Experiment 2 — Circuit generalization and observation choice

> How well does the selected PPO actor generalize from procedurally generated
> training circuits to unseen circuits, and how does the Frenet observation
> compare with local LiDAR sensing?

This notebook is the executable form of the Experiment 2 section of
[`docs/EXPERIMENT.md`](../docs/EXPERIMENT.md). Experiment 1 asked what capacity
does on **one** circuit; this asks what a policy has actually *learned* — whether
it drives this circuit or drives circuits — and whether the answer depends on
how the track is presented to it.

**It depends on Experiment 1.** The PPO actor width is chosen by the rule
recorded in `experiment_1.ipynb`, before any test circuit here is opened. Run
that notebook first.

Two things vary together and must not be confused:

- **Generalization** is a property of one condition: the gap between circuits a
  run trained on and circuits it has never seen.
- **Observation** is the comparison between conditions: Frenet against LiDAR,
  paired within each root.


## Hypotheses

- **Generalization.** PPO trained over generated circuits is expected to retain
  useful performance on unseen generator seeds. Note the protocol's warning: a
  small generalization gap accompanied by poor absolute performance is *not*
  successful generalization, it is uniform failure, and both numbers are
  therefore always reported together.
- **Observation information.** Frenet is expected to learn faster because it
  exposes track-relative geometry and preview curvature directly. LiDAR must
  infer the same things from 16 ranges and may show a larger efficiency or
  final-performance gap.
- **Track variation.** Both conditions can vary substantially with held-out
  geometry, so per-circuit outcomes accompany every root-level summary.

## Design matrix

One independent training unit is $(\text{observation type}, \text{root
identity})$.

| Algorithm | Actor | Observation | Roots |
|---|---|---|---:|
| PPO | selected by the Experiment 1 rule | Frenet | 5 |
| PPO | same hidden widths | LiDAR | 5 |

Ten training runs. Within each root the two runs are paired by training-circuit
schedule, budget, PPO settings, validation circuits and test circuits; only
their mutable RNG objects are separate.

## What each condition observes

$$
O_t^{\mathrm{Frenet}}=(d_t,\phi_{e,t},v_t,\delta_t,\bar\kappa_t)
\qquad
O_t^{\mathrm{LiDAR}}=(v_t,\delta_t,\widetilde r_t^{(1)},\ldots,\widetilde r_t^{(16)})
$$

Both carry speed and steering angle, which are vehicle state rather than
perception; **only the track representation differs**, which is the comparison
the experiment is about. Neither critic receives privileged information, each
condition learns its own normalization statistics from training only and
freezes them for evaluation, and LiDAR stays feed-forward without frame
stacking — so its partial observability is part of the interpretation, not a
defect to be corrected.

The input dimensions differ (5 against 18), so equal hidden widths still give a
small unavoidable parameter-count difference. It is reported rather than
removed.


## Circuit splits

All circuits come from the same frozen generator but from **disjoint
deterministic namespaces**, so a circuit identity in one split can never denote
a circuit in another:

| Split | Count | Role |
|---|---:|---|
| development | 8 | looked at before the experiment; source of the geometry bin edges |
| training | unbounded | drawn per root and per-worker episode |
| validation | 16 | the learning curve and the convergence rule |
| test | 32 | opened once, after training and selection are complete |
| training-reference | 16 | circuits *this run trained on*, revisited |

The splits are committed in `tracks/experiment_2_splits.json`, which stores each
circuit's identity, the generator seed that identity denotes, and its length,
straight fraction, curvature quantiles and tightest radius. **The circuits
themselves are not stored.** They are rebuilt from the frozen generator on
demand and those statistics are re-checked on the way back in, so a change to
the generator becomes a loud failure rather than a silent change in what a
circuit identity means. The check uses a $10^{-6}$ relative tolerance because
generation runs through `math.cos` and `math.sin`, whose last bit belongs to the
platform's maths library; a real change moves the geometry by metres.

**Training-reference circuits are deliberately in-sample.** They are the
exception to the disjoint namespaces: taken in per-worker episode order from
what the run actually raced, so a paired root's two runs revisit the same
circuits. There are 16, matching the validation count, so the two gaps to the
test split rest on the same denominator and can be compared with each other.

## Pairing, and why it is per-worker

Circuits change only at episode reset, and each worker owns an identically
seeded selection stream, so worker $w$ meets the same circuit on its $k$-th
episode in both runs of a paired root. Pairing is by **worker and per-worker
episode count**, never by a global episode index: the two observation policies
produce episodes of different lengths, so their episodes finish in a different
order and a global index would name a different circuit in each run — exactly
the drift the pairing exists to prevent.

The two conditions therefore meet the same circuits in the same per-worker
order, but not necessarily the same *number* of them inside an equal
interaction budget. Circuit exposure is recorded rather than assumed equal.


## Evaluation and the convergence rule

Every validation checkpoint evaluates the deterministic policy once on **each**
of the 16 validation circuits. Stable convergence is the first of three
consecutive checkpoints with:

- validation completion rate at least `0.75`; and
- median validation normalized progress at least `0.95`.

That requires finishing at least 12 of 16 circuits while typical progress stays
near a full lap. Both are project definitions, not values from PPO theory. Runs
that never meet them are censored at the common budget and stay in the summaries.

Note this is a *different* rule from Experiment 1's, which was a lap-time
threshold on one circuit. A single circuit's lap time says nothing about a
distribution of circuits, so the rule changed with the question.

Validation and test interactions never enter the training budget, and
validation updates no network, optimizer, normalizer, schedule or training RNG.
**Test circuits cannot influence training, learning-rate choice, actor-size
selection, convergence or checkpoint selection** — they are opened after the
final policy exists.

## Configuration


In [ ]:
import json
import sys
import warnings
from functools import partial
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning,
    module="pygame.pkgdata",
)

from circuits import CircuitSplit, TrainingCircuitSchedule, load_split_circuits
from configs import (
    LARGE_ACTOR_CONFIG,
    MEDIUM_ACTOR_CONFIG,
    SMALL_ACTOR_CONFIG,
    EnvironmentConfig,
    ExecutionConfig,
    LoggingConfig,
    ObservationRepresentation,
    PPOConfig,
    physical_cpu_count,
)
from matrix import RunSpecification, execute, learning_contract, summarize
from recording import RunCategory
from reporting import describe, read_table, show_figure, show_table
from train import run_ppo_training

# ---------------------------------------------------------------- scale ----
# The only switch in this notebook. A rehearsal exercises every cell of the
# matrix and the whole analysis in minutes; the protocol is 10 runs of two
# million interactions and takes roughly three hours on eight workers.
#
# The run category travels with the choice on purpose. A rehearsal writes under
# `reduced_budget_end_to_end_validation/`, which the recording schema refuses to
# load as reported data, and it draws from a different seed namespace -- so a
# rehearsal can neither be mistaken for a result nor share randomness with one.
REHEARSAL = True

if REHEARSAL:
    TRAINING_INTERACTION_BUDGET = 60_000
    EVALUATION_INTERVAL = 5_000
    ROOTS = (0, 1)
    RUN_CATEGORY = RunCategory.REDUCED_VALIDATION
else:
    TRAINING_INTERACTION_BUDGET = 2_000_000
    EVALUATION_INTERVAL = 50_000
    ROOTS = (0, 1, 2, 3, 4)
    RUN_CATEGORY = RunCategory.REPORTED

# ------------------------------------------------------------- fixtures ----
SPLITS_PATH = PROJECT_ROOT / "tracks" / "experiment_2_splits.json"
RESULTS_ROOT = PROJECT_ROOT / "results" / RUN_CATEGORY.value / "experiment_2"
ANALYSIS_ROOT = PROJECT_ROOT / "results" / "analysis" / RUN_CATEGORY.value / "experiment_2"
EXPERIMENT_1_ANALYSIS = (
    PROJECT_ROOT / "results" / "analysis" / RUN_CATEGORY.value / "experiment_1"
)
EXECUTION_CONFIG = ExecutionConfig(environment_workers=physical_cpu_count())
ENVIRONMENT_CONFIG = EnvironmentConfig()
STEERING_THRESHOLD = LoggingConfig().near_saturated_steering_threshold

# Selected before the experiment; see the configuration check in EXPERIMENT.md.
ACTOR_LEARNING_RATE = 3e-4
CRITIC_LEARNING_RATE = 1e-2
TRAINING_REFERENCE_CIRCUITS = 16

ACTORS = {
    "small": SMALL_ACTOR_CONFIG,
    "medium": MEDIUM_ACTOR_CONFIG,
    "large": LARGE_ACTOR_CONFIG,
}
OBSERVATIONS = {
    "frenet": ObservationRepresentation.FRENET,
    "lidar": ObservationRepresentation.LIDAR,
}
# A finished run is only reused if it was produced under these constants.
CONTRACT = learning_contract(ENVIRONMENT_CONFIG, PPOConfig())

print(f"budget {TRAINING_INTERACTION_BUDGET:,} | roots {ROOTS} | {RUN_CATEGORY.value}")


### The actor width comes from Experiment 1

The rule is not re-derived here. Experiment 1 recorded its calculation, and this
reads the answer. Overriding it is possible but is a deviation from the
protocol, so it is written as an explicit assignment rather than hidden in a
default.


In [ ]:
ACTOR_OVERRIDE = None  # set to "small" / "medium" / "large" only to deviate deliberately

selection_path = EXPERIMENT_1_ANALYSIS / "ppo_actor_selection.json"
if ACTOR_OVERRIDE is not None:
    SELECTED_ACTOR = ACTOR_OVERRIDE
    print(f"OVERRIDDEN: using the {SELECTED_ACTOR!r} actor, not the recorded selection.")
elif selection_path.is_file():
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    SELECTED_ACTOR = selection["selected_actor"]
    show_table(selection["candidates"], title="The recorded Experiment 1 selection")
    print(f"Experiment 1 selected the {SELECTED_ACTOR!r} actor.")
else:
    raise FileNotFoundError(
        f"no recorded actor selection at {selection_path}. "
        "Run experiment_1.ipynb first, at the same run category."
    )

ACTOR_CONFIG = ACTORS[SELECTED_ACTOR]
print(f"hidden widths {ACTOR_CONFIG.hidden_sizes}")


### Rebuilding the frozen splits

Each circuit is regenerated from its identity and its recorded geometry is
re-checked. If the generator has changed, this raises here rather than producing
results that quietly describe different circuits.

The two conditions need **separate circuit objects over the same geometry**. An
evaluation circuit carries a factory that builds its environment, and that
environment produces observations of one type, so a LiDAR policy cannot be
evaluated on a circuit built to emit Frenet observations. The engine checks this
and refuses, which is why the sets are built per observation rather than shared.


In [ ]:
from dataclasses import replace

CIRCUITS = {
    name: {
        split.value: load_split_circuits(
            SPLITS_PATH,
            split,
            environment_config=replace(ENVIRONMENT_CONFIG, observation_type=observation),
        )
        for split in (CircuitSplit.VALIDATION, CircuitSplit.TEST)
    }
    for name, observation in OBSERVATIONS.items()
}
SPLIT_MANIFEST = json.loads(SPLITS_PATH.read_text(encoding="utf-8"))
STRATA = SPLIT_MANIFEST["geometry_strata"]

for name, splits in CIRCUITS.items():
    counts = ", ".join(f"{split} {len(circuits)}" for split, circuits in splits.items())
    print(f"{name:>7}: {counts}")
print(f"geometry bin edges:  {STRATA}")

show_table(
    SPLIT_MANIFEST["splits"]["validation"]["circuits"],
    columns=["identity", "track_seed", "track_length", "straight_fraction"],
    title="Validation split (rebuilt and geometry-verified)",
    limit=6,
)


## Building the matrix

The two conditions of a root differ **only** in the observation. They share the
actor and critic widths, the PPO configuration, the budget, the validation and
test circuits, and — because both derive their selection streams from the same
root — the per-worker sequence of training circuits.

The test circuits are handed to the run as `final_evaluation_circuits`, which
the engine opens only after the last optimizer step and the saved final policy.
Validation at the final budget comes from the scheduled checkpoint that lands
exactly on it, so it is not requested twice.


In [ ]:
def launch_for(observation_name: str, root: int, path: Path):
    """
    Return a no-argument callable that trains one condition of one root.
    """
    return partial(
        run_ppo_training,
        seed=root,
        run_path=path,
        actor_config=ACTOR_CONFIG,
        actor_learning_rate=ACTOR_LEARNING_RATE,
        critic_learning_rate=CRITIC_LEARNING_RATE,
        training_interaction_budget=TRAINING_INTERACTION_BUDGET,
        environment_config=ENVIRONMENT_CONFIG,
        evaluation_interval=EVALUATION_INTERVAL,
        execution_config=EXECUTION_CONFIG,
        near_saturated_steering_threshold=STEERING_THRESHOLD,
        # An unbounded schedule of generated circuits: this is what makes the
        # experiment about circuits rather than about one circuit.
        training_circuit_schedule=TrainingCircuitSchedule(),
        evaluation_circuits=CIRCUITS[observation_name]["validation"],
        final_evaluation_circuits=CIRCUITS[observation_name]["test"],
        training_reference_circuits=TRAINING_REFERENCE_CIRCUITS,
        observation=OBSERVATIONS[observation_name],
        run_category=RUN_CATEGORY,
    )


SPECIFICATIONS = [
    RunSpecification(
        run_id := f"ppo-{SELECTED_ACTOR}-{observation_name}-seed-{root}",
        RESULTS_ROOT / run_id,
        launch_for(observation_name, root, RESULTS_ROOT / run_id),
    )
    for observation_name in OBSERVATIONS
    for root in ROOTS
]

show_table(
    [
        {"run_id": s.run_id, "already complete": (s.path / "completion.json").is_file()}
        for s in SPECIFICATIONS
    ],
    title=f"{len(SPECIFICATIONS)} runs",
)


## Running the matrix

Resumable, contract-checked and failure-tolerant, exactly as in Experiment 1:
a finished run recorded under different constants is re-run rather than reused.
These runs are
slower than Experiment 1's: each validation checkpoint drives 16 deterministic
episodes, and the final policy is evaluated on 16 training-reference, 16
validation and 32 test circuits.


In [ ]:
OUTCOMES = execute(SPECIFICATIONS, contract=CONTRACT)
FAILURES = summarize(OUTCOMES)
assert FAILURES == 0, f"{FAILURES} run(s) failed; see the tracebacks above."


## Analysis

Regenerated from the raw records, with the geometry bin edges taken from the
committed split manifest — never from the results they describe.


In [ ]:
from analyze_results import analyze_results

MANIFEST = analyze_results(
    results_root=RESULTS_ROOT,
    output_directory=ANALYSIS_ROOT,
    experiment=2,
    category=RUN_CATEGORY,
    geometry_specification=SPLITS_PATH,
)
print(f"analyzed {len(MANIFEST['inputs'])} runs -> {ANALYSIS_ROOT}")

INVENTORY = read_table(ANALYSIS_ROOT, "run_inventory")
SUMMARIES = read_table(ANALYSIS_ROOT, "run_summaries")
SPLIT_SUMMARIES = read_table(ANALYSIS_ROOT, "final_split_summaries")
GAPS = read_table(ANALYSIS_ROOT, "generalization_gaps")
PAIRED_CIRCUITS = read_table(ANALYSIS_ROOT, "paired_circuit_differences")
PAIRED = read_table(ANALYSIS_ROOT, "paired_summaries")
GEOMETRY = read_table(ANALYSIS_ROOT, "geometry_strata")


### Parameter counts and circuit exposure

The parameter difference between conditions comes from the input width alone,
and the circuit counts show how much of the generated distribution each
condition actually saw inside the equal interaction budget. The protocol asks
for exposure to be recorded, not assumed equal.


In [ ]:
show_table(
    INVENTORY,
    columns=[
        "run_id", "observation_type", "root_identity",
        "actor_parameters", "critic_parameters", "total_parameters",
        "training_interactions",
    ],
    sort_by=["observation_type", "root_identity"],
    title="Run inventory",
)


### Primary outcome — held-out test performance

Aggregated **within a root across its 32 test circuits first**, then compared
across roots. The 32 circuits are not 32 independently trained policies, and
treating them as such would inflate the sample size by a factor of 32.

Completion rate is the primary number; return and progress follow; and the lap
time always travels with its completion denominator, because a mean lap time
computed only over the circuits a policy happened to finish flatters exactly the
policies that finish least often.


In [ ]:
show_table(
    SPLIT_SUMMARIES,
    columns=[
        "observation_type", "root_identity", "circuit_split", "circuit_count",
        "completion_count", "completion_rate", "crash_rate",
        "mean_return", "mean_progress",
    ],
    sort_by=["circuit_split", "observation_type", "root_identity"],
    where=lambda row: row["circuit_split"] == "test",
    title="Final performance on the 32 unseen test circuits",
)

for observation in ("frenet", "lidar"):
    values = [
        row["completion_rate"]
        for row in SPLIT_SUMMARIES
        if row["circuit_split"] == "test" and row["observation_type"] == observation
    ]
    print(f"{observation:>7} test completion rate: {describe(values)}")


### Generalization — training-reference, validation and test

Three splits, one denominator each of 16, 16 and 32. The gap that matters is
in-sample against held-out; the validation-to-test gap says whether the circuits
used to steer the run were representative of unseen ones.

Read the absolute numbers alongside the gaps. A near-zero gap on a policy that
completes nothing is uniform failure, not generalization.


In [ ]:
show_table(
    SPLIT_SUMMARIES,
    columns=[
        "observation_type", "root_identity", "circuit_split", "circuit_count",
        "completion_rate", "mean_progress", "mean_return",
    ],
    sort_by=["observation_type", "root_identity", "circuit_split"],
    title="Every split, every root",
)
show_table(GAPS, title="Generalization gaps")


### The observation comparison — paired within root

The primary comparison is five paired Frenet-minus-LiDAR differences, one per
root. Pairing removes the root-to-root variation that would otherwise swamp a
five-sample comparison, which is why the design fixes everything except the
observation within a root.


In [ ]:
show_table(PAIRED, title="Paired Frenet-minus-LiDAR root-level differences")
show_table(
    PAIRED_CIRCUITS,
    columns=[
        "root_identity", "circuit_identity", "circuit_split",
        "return", "maximum_progress",
    ],
    sort_by=["root_identity", "circuit_identity"],
    limit=16,
    title="Per-circuit paired differences (first rows; full table on disk)",
)


### Validation learning curves and convergence

Curves align on training interactions. Convergence is the 0.75 completion and
0.95 median-progress rule, with censoring shown rather than averaged away.


In [ ]:
show_figure(ANALYSIS_ROOT, "learning_curves")
show_table(
    SUMMARIES,
    columns=[
        "observation_type", "root_identity", "converged", "censored",
        "convergence_interactions", "convergence_duration",
        "return_auc", "progress_auc",
    ],
    sort_by=["observation_type", "root_identity"],
    title="Convergence and curve area",
)
show_figure(ANALYSIS_ROOT, "convergence_resources")


### Outcomes stratified by circuit geometry

The bin edges are the tertiles of the eight **development** circuits, recorded
in the split commitment before the experiment — chosen from circuits that exist
to be looked at, never from the results they describe.

The curvature statistic is the 90th percentile of absolute sampled curvature,
not the median: over half of every generated circuit is straight, so the median
is exactly zero everywhere and separates nothing.


In [ ]:
show_table(GEOMETRY, title="Completion and progress by length and curvature bin")
show_figure(ANALYSIS_ROOT, "circuit_geometry")
show_figure(ANALYSIS_ROOT, "curvature_controls")
show_figure(ANALYSIS_ROOT, "task_outcomes")


### Computation and optimization diagnostics

LiDAR carries 18 inputs against Frenet's 5, so its first layer is wider and its
per-step cost differs. Whether that shows up as throughput or as optimization
time is worth separating.


In [ ]:
show_table(
    SUMMARIES,
    columns=[
        "observation_type", "root_identity", "collection_throughput",
        "collection_duration", "optimization_duration", "evaluation_duration",
        "end_to_end_duration", "peak_process_memory",
    ],
    sort_by=["observation_type", "root_identity"],
    title="Computational cost",
)
show_figure(ANALYSIS_ROOT, "optimization_diagnostics")


## Limitations

Stated by the protocol, and unchanged by any result above:

- Five training roots provide modest evidence about optimizer randomness.
- Test circuits cover only the frozen procedural generator's distribution;
  nothing here speaks to circuits it cannot produce.
- Feed-forward LiDAR is intentionally partially observable and may be
  disadvantaged relative to a recurrent sensor policy. That is a property of
  this comparison, not a finding about LiDAR sensing in general.
- Different episode lengths produce different total circuit exposure within an
  equal interaction budget, which is recorded above rather than corrected.
- Input dimensions create a small unavoidable parameter-count difference even at
  equal hidden widths.
